In [ ]:
import numpy as np
import torch

from giotto.agents.algorithms.alphazero.mcts import AlphaZeroMCTS, AZNode
from giotto.agents.algorithms.alphazero.net import AlphaZeroNet
from giotto.envs.tris import TrisEnv
from giotto.utils.random_board import random_board

# Neural Network
Two headed network that takes encoded board input and outputs policy logits and estimated position value

In [ ]:
state, env = random_board()
env.render()

print(state)

In [ ]:
env.get_state()

In [ ]:
net = AlphaZeroNet(input_size=[2, 3, 3], policy_output_size=9, value_output_size=1, channels=64, residual_blocks=5)

input = net.process_state(state)
print(input.shape)
print(input.numpy())

In [ ]:
with torch.no_grad():
    policy, value = net.forward(input)

print(policy.numpy())
print(torch.nn.functional.softmax(policy, dim=1).numpy())
print(value.numpy())

In [ ]:
policy, value = net.predict(state)

print(policy)
print(value)

# Monte Carlo Tree Search

In [ ]:
env = TrisEnv()

# mock game
env.step(1)
env.step(5)
env.step(9)
env.step(3)
env.step(7)
env.step(8)
env.render()

# mock policy and value
# policy = np.array([-10.0, 0.1, -10.0, 1.0, -10.0, 0.1, -10.0, -10.0, -10.0])
# value = 0.85

In [ ]:
dirichlet_alpha = 1.0
dirichlet_eps = 0.0
cpuct = 1.5
n_simulations = 100
temperature = 0.0  # greedy

In [ ]:
root_env = env.clone()
root = AZNode(env=root_env, parent=None, parent_action=None)
# ROOT EXPANSION
# expand root, adding dirichlet noise to each action
valid_actions = root_env.get_valid_actions()
policy, value = net.predict(root_env.get_state())
noise = np.random.dirichlet([dirichlet_alpha] * 9)
action_probs = ((1 - dirichlet_eps) * policy) + dirichlet_eps * noise

# valid action masking
action_probs = [action_probs[a - 1] for a in valid_actions]
action_probs /= np.sum(action_probs)  # reapply softmax

for action, prob in zip(valid_actions, action_probs, strict=True):
    new_env = root_env.clone()
    new_env.step(action)
    root.children[action] = AZNode(env=new_env, parent=root, parent_action=action)
    root.children[action].prob = prob
root.n_visits = 1
root.total_score = value

print(valid_actions)
print(action_probs)
print(root)

In [ ]:
def select_action(root: AZNode, temperature: float = 0.0):
    """Select an action from the root based on visit counts, adjusted by temperature. Set 0 for greedy selection."""
    action_counts = {key: val.n_visits for key, val in root.children.items()}
    if temperature == 0:
        return max(action_counts, key=action_counts.get)
    elif temperature == np.inf:
        return np.random.choice(list(action_counts.keys()))
    else:
        distribution = np.array([*action_counts.values()]) ** (1 / temperature)
        return np.random.choice([*action_counts.keys()], p=distribution / sum(distribution))


# SEARCH
for _ in range(n_simulations):
    current_node = root

    # SELECTION
    while not current_node.is_terminal() and not current_node.is_leaf():
        current_node = current_node.select_child(cpuct=cpuct)

    # EXPANSION
    if not current_node.is_terminal():
        current_node.expand()
        policy, value = net.predict(current_node.env.get_state())
        valid_actions = current_node.env.get_valid_actions()
        action_probs = [policy[a - 1] for a in valid_actions]
        action_probs /= np.sum(action_probs)  # reapply softmax

        for action, prob in zip(valid_actions, action_probs, strict=True):
            current_node.children[action].prob = prob
    # if node is terminal, get the value of it from env
    else:
        value = current_node.terminal_node_eval(env=current_node.env, player=current_node.to_play)

    # BACKPROPAGATION
    current_node.backpropagate(value)

# select action (temperature regulates exploration)
print(f"Selected action: {select_action(root, temperature)}")
print(root)

print("-------------")
print("Root children:")
for parent_action, child in root.children.items():
    print(f"Parent action: {parent_action}")
    print(child)

In [ ]:
# same as running the MCTS

mcts = AlphaZeroMCTS(net=net, n_simulations=100, cpuct=1.5, dirichlet_alpha=1, dirichlet_eps=0, temperature=0)

action, root = mcts.run(env)

print(f"Selected action: {action}")
print(f"Position estimated value {root.avg_value}")